# Laubach (35321) Full Municipality Rural Roads Analysis

This notebook uses the **administrative municipal boundary** of Laubach (ZIP 35321, including all villages) as the area of interest.

It performs:
- Administrative boundary extraction from OSM
- Rural road extraction from OSM
- OSM-based road characterisation
- Advanced statistics (length, density, composition, uncertainty flags)
- Spatial summaries by settlement/village context
- Static and interactive mapping
- Export of GeoPackage, CSV, and HTML map outputs

In [ ]:
# Optional first-run install (uncomment if needed)
# %pip install osmnx folium mapclassify

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import folium
import osmnx as ox

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
ox.settings.use_cache = True
ox.settings.log_console = False

In [ ]:
# Configuration
PLACE_QUERY = "Laubach, Landkreis Giessen, Hessen, Germany"
OUTPUT_DIR = Path("analysis_outputs/laubach_35321_admin")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CRS_METRIC = "EPSG:25832"  # ETRS89 / UTM zone 32N

# Rural-road focused OSM highway classes
RURAL_HIGHWAY_TYPES = [
    "track",
    "path",
    "service",
    "unclassified",
    "tertiary",
    "tertiary_link",
    "residential"
]

# Characterisation groups
SEALED_SURFACES = {"asphalt", "paved", "concrete", "paving_stones", "sett"}
MINERAL_SURFACES = {"compacted", "gravel", "fine_gravel", "pebblestone", "rock", "unpaved"}
NATURAL_SURFACES = {"dirt", "earth", "ground", "grass", "mud", "sand", "clay"}

In [ ]:
# 1) Administrative municipal boundary from OSM
admin = ox.geocode_to_gdf(PLACE_QUERY)
admin = admin[["display_name", "osm_type", "osm_id", "geometry"]].copy()
admin = admin.to_crs("EPSG:4326")

if len(admin) > 1:
    admin = admin.iloc[[0]].copy()

admin_poly = admin.geometry.iloc[0]
admin_metric = admin.to_crs(TARGET_CRS_METRIC)
admin_area_km2 = admin_metric.geometry.area.iloc[0] / 1_000_000

print(f"Boundary source: {admin.display_name.iloc[0]}")
print(f"OSM id: {admin.osm_type.iloc[0]}/{admin.osm_id.iloc[0]}")
print(f"Administrative area: {admin_area_km2:,.2f} km^2")

In [ ]:
# 2) Settlement context (villages/hamlets/suburbs) inside municipal boundary
settlement_tags = {"place": ["village", "hamlet", "suburb", "neighbourhood"]}
settlements = ox.features_from_polygon(admin_poly, tags=settlement_tags).reset_index()

keep_cols = [c for c in ["name", "place", "population", "geometry"] if c in settlements.columns]
settlements = settlements[keep_cols].copy()
settlements = settlements.dropna(subset=["geometry"]).to_crs("EPSG:4326")

# Keep only point-like references for village labeling and deduplicate by name
is_point = settlements.geometry.geom_type.isin(["Point", "MultiPoint"])
settlements_points = settlements[is_point].copy()
if "name" in settlements_points.columns:
    settlements_points = settlements_points.sort_values("name").drop_duplicates(subset=["name", "place"])

print(f"Detected settlement references inside boundary: {len(settlements_points)}")
display(settlements_points[[c for c in ["name", "place", "population"] if c in settlements_points.columns]].head(20))

In [ ]:
# 3) Extract rural roads from OSM (within municipality boundary)
road_tags = {"highway": RURAL_HIGHWAY_TYPES}
roads_raw = ox.features_from_polygon(admin_poly, tags=road_tags).reset_index()

# Keep only line geometries
roads = roads_raw[roads_raw.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()

wanted_cols = [
    "osmid", "name", "highway", "surface", "tracktype", "smoothness",
    "access", "service", "maxspeed", "lit", "geometry"
]
keep_cols = [c for c in wanted_cols if c in roads.columns]
roads = roads[keep_cols].copy()
roads = roads.explode(index_parts=False).reset_index(drop=True)
roads = roads.to_crs("EPSG:4326")

print(f"Extracted rural-road segments: {len(roads):,}")
roads.head(3)

In [ ]:
# 4) Characterise roads using OSM attributes + geometry
def to_scalar(v):
    if isinstance(v, list):
        return v[0] if len(v) else np.nan
    return v

for col in ["highway", "surface", "tracktype", "smoothness", "access", "service", "maxspeed", "lit", "name"]:
    if col in roads.columns:
        roads[col] = roads[col].apply(to_scalar)

roads_m = roads.to_crs(TARGET_CRS_METRIC)
roads["length_m"] = roads_m.geometry.length

def surface_group(surface):
    if pd.isna(surface):
        return "unknown"
    s = str(surface).lower().strip()
    if s in SEALED_SURFACES:
        return "sealed"
    if s in MINERAL_SURFACES:
        return "mineral"
    if s in NATURAL_SURFACES:
        return "natural"
    return "other"

def accessibility_class(access):
    if pd.isna(access):
        return "unspecified"
    a = str(access).lower().strip()
    if a in {"yes", "permissive", "destination"}:
        return "open_or_limited"
    if a in {"private", "no", "agricultural", "forestry"}:
        return "restricted"
    return "other"

roads["surface_group"] = roads.get("surface", pd.Series(index=roads.index)).apply(surface_group)
roads["access_group"] = roads.get("access", pd.Series(index=roads.index)).apply(accessibility_class)
roads["is_track_like"] = roads.get("highway", pd.Series(index=roads.index)).astype(str).isin(["track", "path"])
roads["uncertain_surface"] = roads["surface_group"].isin(["unknown", "other"])

roads.head(5)

In [ ]:
# 5) Advanced descriptive statistics
total_len_km = roads["length_m"].sum() / 1000
road_density_km_per_km2 = total_len_km / admin_area_km2

stats_overview = pd.DataFrame({
    "metric": [
        "administrative_area_km2",
        "rural_segments_count",
        "rural_length_km",
        "rural_density_km_per_km2",
        "median_segment_length_m",
        "p90_segment_length_m",
        "unknown_or_other_surface_share"
    ],
    "value": [
        admin_area_km2,
        len(roads),
        total_len_km,
        road_density_km_per_km2,
        roads["length_m"].median(),
        roads["length_m"].quantile(0.90),
        roads["uncertain_surface"].mean()
    ]
})

display(stats_overview)

length_by_highway = (
    roads.groupby("highway", dropna=False)["length_m"]
    .sum()
    .sort_values(ascending=False)
    .rename("length_m")
    .to_frame()
)
length_by_highway["share"] = length_by_highway["length_m"] / length_by_highway["length_m"].sum()

length_by_surface = (
    roads.groupby("surface_group", dropna=False)["length_m"]
    .sum()
    .sort_values(ascending=False)
    .rename("length_m")
    .to_frame()
)
length_by_surface["share"] = length_by_surface["length_m"] / length_by_surface["length_m"].sum()

display(length_by_highway)
display(length_by_surface)

In [ ]:
# 6) Settlement-level proximity profile
if len(settlements_points) > 0:
    roads_metric = roads.to_crs(TARGET_CRS_METRIC)
    settlements_metric = settlements_points.to_crs(TARGET_CRS_METRIC)

    nearest = gpd.sjoin_nearest(
        roads_metric[["geometry", "length_m", "surface_group", "highway"]],
        settlements_metric[["name", "place", "geometry"]],
        how="left",
        distance_col="dist_to_settlement_m"
    )

    by_settlement = (
        nearest.groupby(["name", "place"], dropna=False)
        .agg(
            n_segments=("length_m", "size"),
            length_km=("length_m", lambda s: s.sum() / 1000),
            median_dist_m=("dist_to_settlement_m", "median"),
            p90_dist_m=("dist_to_settlement_m", lambda s: np.nanquantile(s, 0.9))
        )
        .sort_values("length_km", ascending=False)
        .reset_index()
    )

    display(by_settlement.head(15))
else:
    by_settlement = pd.DataFrame()
    print("No point-based settlements were found for settlement-level profiling.")

In [ ]:
# 7) Network analysis for selected rural road classes
custom_filter = (
    '["highway"~"track|path|service|unclassified|tertiary|tertiary_link|residential"]'
)
G = ox.graph_from_polygon(admin_poly, custom_filter=custom_filter, simplify=True)

nodes_gdf, edges_gdf = ox.graph_to_gdfs(G, nodes=True, edges=True, fill_edge_geometry=True)
edges_gdf = edges_gdf.to_crs(TARGET_CRS_METRIC)

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
edge_len_km = edges_gdf.length.sum() / 1000
components = list(nx.weakly_connected_components(G))
largest_component_share = max(len(c) for c in components) / n_nodes if n_nodes > 0 else np.nan

degrees = [d for _, d in G.degree()]
network_stats = pd.DataFrame({
    "metric": ["nodes", "edges", "edge_length_km", "avg_degree", "largest_component_node_share"],
    "value": [n_nodes, n_edges, edge_len_km, np.mean(degrees), largest_component_share]
})
display(network_stats)

In [ ]:
# 8) Statistical plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Segment length distribution
sns.histplot(roads["length_m"], bins=40, ax=axes[0, 0], color="#2a9d8f")
axes[0, 0].set_title("Segment length distribution (m)")
axes[0, 0].set_xlabel("Length (m)")

# Length by highway class
length_by_highway_plot = roads.groupby("highway")["length_m"].sum().sort_values(ascending=False) / 1000
sns.barplot(x=length_by_highway_plot.values, y=length_by_highway_plot.index, ax=axes[0, 1], color="#264653")
axes[0, 1].set_title("Rural road length by highway class (km)")
axes[0, 1].set_xlabel("Length (km)")

# Surface composition by length
surface_comp = roads.groupby("surface_group")["length_m"].sum().sort_values(ascending=False)
axes[1, 0].pie(surface_comp.values, labels=surface_comp.index, autopct="%1.1f%%", startangle=90)
axes[1, 0].set_title("Surface-group composition (length-weighted)")

# Degree distribution
sns.histplot(degrees, bins=20, ax=axes[1, 1], color="#e76f51")
axes[1, 1].set_title("Network node degree distribution")
axes[1, 1].set_xlabel("Node degree")

plt.tight_layout()
plot_path = OUTPUT_DIR / "statistical_overview.png"
plt.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved plot panel: {plot_path}")

In [ ]:
# 9) Interactive mapping (Folium)
admin_centroid = admin.geometry.iloc[0].centroid
m = folium.Map(location=[admin_centroid.y, admin_centroid.x], zoom_start=12, tiles="CartoDB positron")

# Municipal boundary
folium.GeoJson(
    admin.to_json(),
    name="Administrative boundary",
    style_function=lambda _: {"color": "#1d3557", "weight": 3, "fillOpacity": 0.02}
).add_to(m)

# Roads styled by surface_group
surface_colors = {
    "sealed": "#3d405b",
    "mineral": "#bc6c25",
    "natural": "#588157",
    "unknown": "#6c757d",
    "other": "#6c757d"
}

roads_map = roads[[c for c in ["highway", "surface", "surface_group", "length_m", "geometry"] if c in roads.columns]].copy()
roads_map["length_m"] = roads_map["length_m"].round(1)

folium.GeoJson(
    roads_map.to_json(),
    name="Rural roads",
    style_function=lambda feature: {
        "color": surface_colors.get(feature["properties"].get("surface_group", "unknown"), "#6c757d"),
        "weight": 2
    },
    tooltip=folium.GeoJsonTooltip(fields=["highway", "surface", "surface_group", "length_m"], aliases=["highway", "surface", "surface_group", "length_m"])
).add_to(m)

# Settlements
if len(settlements_points) > 0:
    for _, row in settlements_points.iterrows():
        geom = row.geometry
        folium.CircleMarker(
            location=[geom.y, geom.x],
            radius=4,
            color="#d62828",
            fill=True,
            fill_opacity=0.85,
            tooltip=f"{row.get('name', 'Unnamed')} ({row.get('place', 'n/a')})"
        ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
# 10) Export outputs
admin_out = OUTPUT_DIR / "laubach_35321_admin_boundary.gpkg"
roads_out = OUTPUT_DIR / "laubach_35321_rural_roads.gpkg"
stats_out = OUTPUT_DIR / "laubach_35321_stats_overview.csv"
highway_out = OUTPUT_DIR / "laubach_35321_length_by_highway.csv"
surface_out = OUTPUT_DIR / "laubach_35321_length_by_surface_group.csv"
settlement_out = OUTPUT_DIR / "laubach_35321_settlement_profile.csv"
map_out = OUTPUT_DIR / "laubach_35321_rural_roads_map.html"

admin.to_file(admin_out, layer="admin_boundary", driver="GPKG")
roads.to_file(roads_out, layer="rural_roads", driver="GPKG")
stats_overview.to_csv(stats_out, index=False)
length_by_highway.reset_index().to_csv(highway_out, index=False)
length_by_surface.reset_index().to_csv(surface_out, index=False)

if len(by_settlement) > 0:
    by_settlement.to_csv(settlement_out, index=False)

m.save(map_out)

print(f"Saved: {admin_out}")
print(f"Saved: {roads_out}")
print(f"Saved: {stats_out}")
print(f"Saved: {highway_out}")
print(f"Saved: {surface_out}")
if len(by_settlement) > 0:
    print(f"Saved: {settlement_out}")
print(f"Saved: {map_out}")

## Notes

- The exact list/count of villages depends on current OSM tagging quality and geometry type (point vs polygon references).
- Surface and accessibility characterization is OSM-attribute based and may be incomplete in segments with sparse tags.
- You can tighten the rural-road definition by removing `residential` and/or `tertiary` from `RURAL_HIGHWAY_TYPES`.
- For remote-sensing enrichment, merge this output with your openEO feature table (`openeo_outputs/laubach_feature_table.csv`) on OSM id / geometry matching.